In [1]:
# Imports

In [2]:
import warnings

warnings.filterwarnings("ignore")
import numpy as np
from torchvision import transforms
import os

import torch
import argparse
from dataset.dataset_ec import ECdataset

from sklearn.metrics import f1_score
from time import time
from utils.load_weights import *
from utils.sam import SAM

from models.FSFNet import FSFNet
from dataset.randomaug import RandAugment

In [3]:
import random
def set_random_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    np.random.seed(seed)  # Numpy module.
    random.seed(seed)  # Python random module.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

## Load Dataset

In [5]:
data_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((112, 112)),
    RandAugment(1,5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(scale=(0.02, 0.1)),
])

data_transforms_val = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

In [6]:
num_classes = 4
datapath = '/Users/giulia_huang/Desktop/Annotations/'
train_dataset = ECdataset(datapath, train=True, transform=data_transforms)
val_dataset = ECdataset(datapath, train=False, transform=data_transforms_val)

Check names  ['1_54_rec03_pos3_video_0540_0550' '1_100_rec03_pos1_video_1000_1010'
 '2_19_rec03_pos2_video_0190_0200' '2_73_rec01_pos4_video_0730_0740'
 '2_43_rec02_pos4_video_0430_0440' '2_97_rec02_pos4_video_0970_0980'
 '1_5_rec03_pos2_video_0050_0060' '1_27_rec01_pos1_video_0270_0280'
 '2_3_rec01_pos4_video_0030_0040' '1_39_rec01_pos1_video_0390_0400']
Check target [0 3 0 0 0 0 0 0 0 2]
Check names  ['2_17_rec01_pos1_video_0170_0180' '1_10_rec01_pos4_video_0100_0110'
 '1_123_rec03_pos4_video_1230_1240' '2_18_rec01_pos3_video_0180_0190'
 '2_58_rec02_pos4_video_0580_0590' '2_100_rec01_pos3_video_1000_1010'
 '1_65_rec03_pos4_video_0650_0660' '2_100_rec02_pos4_video_1000_1010'
 '2_24_rec03_pos2_video_0240_0250' '2_70_rec02_pos3_video_0700_0710']
Check target [2 0 3 0 0 0 0 0 0 0]


In [7]:
num_classes = 4
modeltype = 'large'
model = FSFNet(img_size=112, num_classes=num_classes, type=modeltype)

loading pretrained model
load_weight 0


## Load Labels

## Load Model

In [7]:
print('Validation set size:', val_dataset.__len__())

Validation set size: 2352


In [9]:
sample, label = val_dataset[0]
val_dataset

/Users/giulia_huang/Desktop/Annotations/dataset/1_0_rec01_pos3_video_0000_0010.bmp


In [10]:
for batch_i, (imgs, targets) in enumerate(val_loader):
    print(imgs.size(), targets)
    break

NameError: name 'val_loader' is not defined

In [9]:
batch_size = 128
num_workers= 2
val_num = val_dataset.__len__()

In [10]:
val_num = val_dataset.__len__()
print('Train set size:', train_dataset.__len__())
print('Validation set size:', val_dataset.__len__())

train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=batch_size,
                                           num_workers=num_workers,
                                           shuffle=True,
                                           pin_memory=True)


val_loader = torch.utils.data.DataLoader(val_dataset,
                                         batch_size=batch_size,
                                         num_workers=num_workers,
                                         shuffle=False,
                                         pin_memory=True)

Train set size: 1881
Validation set size: 471


In [11]:
model.to("cpu")

FSFNet(
  (backbone): Backbone(
    (input_layer): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): PReLU(num_parameters=64)
    )
    (body): Sequential(
      (0): bottleneck(
        (shortcut_layer): MaxPool2d(kernel_size=1, stride=2, padding=0, dilation=1, ceil_mode=False)
        (res_layer): Sequential(
          (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (2): PReLU(num_parameters=64)
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (1): bottleneck(
        (shortcut_layer): MaxPool2d(kernel_size=1, stride=1, padding=0, dilati

In [12]:
CE_criterion = torch.nn.CrossEntropyLoss()

In [16]:
pre_labels = []
gt_labels = []
with torch.no_grad():
    val_loss = 0.0
    iter_cnt = 0
    bingo_cnt = 0
    model.eval()
    for batch_i, (imgs, targets) in enumerate(val_loader):
        outputs, features = model(imgs.to("cpu"))
        targets = targets.to("cpu")

        CE_loss = CE_criterion(outputs, targets)
        loss = CE_loss

        val_loss += loss
        iter_cnt += 1
        _, predicts = torch.max(outputs, 1)
        correct_or_not = torch.eq(predicts, targets)
        bingo_cnt += correct_or_not.sum().cpu()
        pre_labels += predicts.cpu().tolist()
        gt_labels += targets.cpu().tolist()

val_loss = val_loss / iter_cnt
val_acc = bingo_cnt.float() / float(val_num)
val_acc = np.around(val_acc.numpy(), 4)
f1 = f1_score(pre_labels, gt_labels, average='macro')
total_socre = 0.67 * f1 + 0.33 * val_acc

print("Validation accuracy:%.4f, Loss:%.3f, f1 %4f, score %4f" % (val_acc, val_loss, f1, total_socre))

/Users/giulia_huang/Desktop/Annotations/dataset/1_0_rec01_pos3_video_0000_0010.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_1_rec01_pos3_video_0010_0020.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_2_rec01_pos3_video_0020_0030.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_3_rec01_pos3_video_0030_0040.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_4_rec01_pos3_video_0040_0050.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_5_rec01_pos3_video_0050_0060.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_6_rec01_pos3_video_0060_0070.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_7_rec01_pos3_video_0070_0080.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_8_rec01_pos3_video_0080_0090.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_9_rec01_pos3_video_0090_0100.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_10_rec01_pos3_video_0100_0110.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_11_rec01_pos3_video_0110_0120.bmp
/U

/Users/giulia_huang/Desktop/Annotations/dataset/1_25_rec01_pos2_video_0250_0260.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_26_rec01_pos2_video_0260_0270.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_27_rec01_pos2_video_0270_0280.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_28_rec01_pos2_video_0280_0290.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_29_rec01_pos2_video_0290_0300.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_30_rec01_pos2_video_0300_0310.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_31_rec01_pos2_video_0310_0320.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_32_rec01_pos2_video_0320_0330.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_33_rec01_pos2_video_0330_0340.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_34_rec01_pos2_video_0340_0350.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_35_rec01_pos2_video_0350_0360.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/1_36_rec01_pos2_video_0360_0

/Users/giulia_huang/Desktop/Annotations/dataset/2_10_rec03_pos1_video_0100_0110.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_11_rec03_pos1_video_0110_0120.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_12_rec03_pos1_video_0120_0130.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_13_rec03_pos1_video_0130_0140.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_14_rec03_pos1_video_0140_0150.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_15_rec03_pos1_video_0150_0160.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_16_rec03_pos1_video_0160_0170.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_17_rec03_pos1_video_0170_0180.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_18_rec03_pos1_video_0180_0190.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_19_rec03_pos1_video_0190_0200.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_20_rec03_pos1_video_0200_0210.bmp
/Users/giulia_huang/Desktop/Annotations/dataset/2_21_rec03_pos1_video_0210_0

Validation accuracy:0.2381, Loss:1.449, f1 0.197028, score 0.210582


In [17]:
import torch
import numpy as np
from sklearn.metrics import f1_score

pre_labels = []
gt_labels = []
val_loss = 0.0
correct_predictions = 0
total_samples = len(val_dataset)

model.eval()
with torch.no_grad():
    for batch_i, (imgs, targets) in enumerate(val_loader):
        imgs, targets = imgs.to("cpu"), targets.to("cpu")  # Move to CPU

        outputs, features = model(imgs)  # Get predictions

        # Compute loss
        loss = CE_criterion(outputs, targets)
        val_loss += loss.item()

        # Predictions
        predicts = torch.argmax(outputs, dim=1)  # More efficient
        correct_predictions += torch.eq(predicts, targets).sum().item()

        # Store labels for F1 score calculation
        pre_labels.extend(predicts.tolist())
        gt_labels.extend(targets.tolist())

# Compute final metrics
val_loss /= len(val_loader)  # Average loss over batches
val_acc = correct_predictions / total_samples  # Compute accuracy
f1 = f1_score(gt_labels, pre_labels, average="macro")  # F1 Score
total_score = 0.67 * f1 + 0.33 * val_acc  # Weighted score

print(f"[Epoch {i}] Validation accuracy: {val_acc:.4f}, Loss: {val_loss:.3f}, F1: {f1:.4f}, Score: {total_score:.4f}")

NameError: name 'i' is not defined

In [18]:
print(f"Validation accuracy: {val_acc:.4f}, Loss: {val_loss:.3f}, F1: {f1:.4f}, Score: {total_score:.4f}")

Validation accuracy: 0.2381, Loss: 1.449, F1: 0.1970, Score: 0.2106


## Fine Tune

In [42]:
for name, param in model.named_parameters():
    param.requires_grad = False  # Freeze everything

# Unfreeze classification head
for name, param in model.ca.named_parameters():
    param.requires_grad = True
    print(f"Unfreezing: ",name)
    
for name, param in model.head.named_parameters():
    param.requires_grad = True
    print(f"Unfreezing: ",name)
    
for name, param in model.fsf_encoder.named_parameters():
    param.requires_grad = True
    print(f"Unfreezing: ",name)

Unfreezing:  linear1.weight
Unfreezing:  linear1.bias
Unfreezing:  linear2.weight
Unfreezing:  linear2.bias
Unfreezing:  linear.weight
Unfreezing:  linear.bias
Unfreezing:  pos_embed
Unfreezing:  blocks.0.norm1.weight
Unfreezing:  blocks.0.norm1.bias
Unfreezing:  blocks.0.attn.qkv.weight
Unfreezing:  blocks.0.attn.qkv.bias
Unfreezing:  blocks.0.attn.proj.weight
Unfreezing:  blocks.0.attn.proj.bias
Unfreezing:  blocks.0.norm2.weight
Unfreezing:  blocks.0.norm2.bias
Unfreezing:  blocks.0.mlp.fc1.weight
Unfreezing:  blocks.0.mlp.fc1.bias
Unfreezing:  blocks.0.mlp.fc2.weight
Unfreezing:  blocks.0.mlp.fc2.bias
Unfreezing:  blocks.1.norm1.weight
Unfreezing:  blocks.1.norm1.bias
Unfreezing:  blocks.1.attn.qkv.weight
Unfreezing:  blocks.1.attn.qkv.bias
Unfreezing:  blocks.1.attn.proj.weight
Unfreezing:  blocks.1.attn.proj.bias
Unfreezing:  blocks.1.norm2.weight
Unfreezing:  blocks.1.norm2.bias
Unfreezing:  blocks.1.mlp.fc1.weight
Unfreezing:  blocks.1.mlp.fc1.bias
Unfreezing:  blocks.1.mlp.fc2

In [43]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from time import time
from sklearn.metrics import f1_score

lr = 0.000004

# Define optimizer & scheduler
base_optimizer = torch.optim.SGD
optimizer = SAM(model.parameters(), base_optimizer, lr=0.000004, rho=0.05, adaptive=False)  # Smaller LR for fine-tuning
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)
CE_criterion = nn.CrossEntropyLoss()

In [44]:
epochs = 100

In [45]:
# Training loop
best_acc = 0
for i in range(1, epochs + 1):
    
    train_loss = 0.0
    correct_sum = 0
    iter_cnt = 0
    start_time = time()
    
    model.train()
    print("Epoch: ", i)
    for batch_i, (imgs, targets) in enumerate(train_loader):
        iter_cnt += 1
        optimizer.zero_grad()
        imgs, targets = imgs.to("cpu"), targets.to("cpu")
        
        outputs, features = model(imgs)
        CE_loss = CE_criterion(outputs, targets)
        loss = CE_loss
        loss.backward()
        optimizer.first_step(zero_grad=True)

        # Second forward-backward pass
        outputs, features = model(imgs)
        CE_loss = CE_criterion(outputs, targets)
        loss = CE_loss
        loss.backward()
        optimizer.second_step(zero_grad=True)

        train_loss += loss
        _, predicts = torch.max(outputs, 1)
        correct_sum += torch.eq(predicts, targets).sum()

    train_acc = correct_sum.float() / float(len(train_dataset))
    train_loss /= iter_cnt
    elapsed = (time() - start_time) / 60

    print(f"[Epoch {i}] Train time: {elapsed:.2f}, Training accuracy: {train_acc:.4f}, Loss: {train_loss:.3f}, LR: {optimizer.param_groups[0]['lr']:.6f}")
    
#    with open(log_txt_path, 'a') as f:
#        f.write(f"[Epoch {i}] Train accuracy: {train_acc:.4f}, Loss: {train_loss:.3f}, LR: {optimizer.param_groups[0]['lr']:.6f}\n")

    scheduler.step()

    # Validation
    pre_labels, gt_labels = [], []
    with torch.no_grad():
        val_loss, iter_cnt, bingo_cnt = 0.0, 0, 0
        model.eval()
        for batch_i, (imgs, targets) in enumerate(val_loader):
            imgs, targets = imgs.to("cpu"), targets.to("cpu")
            outputs, features = model(imgs)

            CE_loss = CE_criterion(outputs, targets)
            val_loss += CE_loss
            iter_cnt += 1

            _, predicts = torch.max(outputs, 1)
            bingo_cnt += torch.eq(predicts, targets).sum().cpu()
            pre_labels += predicts.cpu().tolist()
            gt_labels += targets.cpu().tolist()

        val_loss /= iter_cnt
        val_acc = bingo_cnt.float() / float(val_num)
        val_acc = np.around(val_acc.numpy(), 4)
        f1 = f1_score(pre_labels, gt_labels, average='macro')
        total_score = 0.67 * f1 + 0.33 * val_acc

        print(f"[Epoch {i}] Validation accuracy: {val_acc:.4f}, Loss: {val_loss:.3f}, F1: {f1:.4f}, Score: {total_score:.4f}")

#        with open(log_txt_path, 'a') as f:
#            f.write(f"[Epoch {i}] Validation accuracy: {val_acc:.4f}\n")

        # Save best model
        if val_acc > best_acc:
            torch.save({'epoch': i, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()},
                       os.path.join('./checkpoint', "best_finetuned.pth"))
            print('Fine-tuned model saved.')

#            with open(log_txt_path, 'a') as f:
#                f.write("Fine-tuned model saved.\n")

            best_acc = val_acc
            print("New best validation accuracy:", best_acc)

print("Final best accuracy:", best_acc)

Epoch:  1
[Epoch 1] Train time: 6.07, Training accuracy: 0.1239, Loss: 1.624, LR: 0.000004
[Epoch 1] Validation accuracy: 0.3057, Loss: 1.414, F1: 0.2007, Score: 0.2353
Fine-tuned model saved.
New best validation accuracy: 0.3057
Epoch:  2
[Epoch 2] Train time: 5.95, Training accuracy: 0.1260, Loss: 1.622, LR: 0.000004
[Epoch 2] Validation accuracy: 0.3036, Loss: 1.409, F1: 0.1961, Score: 0.2316
Epoch:  3
[Epoch 3] Train time: 5.99, Training accuracy: 0.1414, Loss: 1.628, LR: 0.000004
[Epoch 3] Validation accuracy: 0.3057, Loss: 1.406, F1: 0.1924, Score: 0.2298
Epoch:  4
[Epoch 4] Train time: 5.95, Training accuracy: 0.1329, Loss: 1.634, LR: 0.000004
[Epoch 4] Validation accuracy: 0.3057, Loss: 1.404, F1: 0.1906, Score: 0.2286
Epoch:  5
[Epoch 5] Train time: 5.98, Training accuracy: 0.1425, Loss: 1.615, LR: 0.000004
[Epoch 5] Validation accuracy: 0.3121, Loss: 1.404, F1: 0.1979, Score: 0.2356
Fine-tuned model saved.
New best validation accuracy: 0.3121
Epoch:  6
[Epoch 6] Train time: 5

[Epoch 45] Validation accuracy: 0.3631, Loss: 1.372, F1: 0.2029, Score: 0.2558
Fine-tuned model saved.
New best validation accuracy: 0.3631
Epoch:  46
[Epoch 46] Train time: 5.43, Training accuracy: 0.1430, Loss: 1.576, LR: 0.000002
[Epoch 46] Validation accuracy: 0.3482, Loss: 1.369, F1: 0.2098, Score: 0.2554
Epoch:  47
[Epoch 47] Train time: 5.44, Training accuracy: 0.1584, Loss: 1.585, LR: 0.000002
[Epoch 47] Validation accuracy: 0.3291, Loss: 1.375, F1: 0.1996, Score: 0.2423
Epoch:  48
[Epoch 48] Train time: 5.44, Training accuracy: 0.1558, Loss: 1.576, LR: 0.000002
[Epoch 48] Validation accuracy: 0.3588, Loss: 1.377, F1: 0.2210, Score: 0.2665
Epoch:  49
[Epoch 49] Train time: 5.44, Training accuracy: 0.1611, Loss: 1.582, LR: 0.000002
[Epoch 49] Validation accuracy: 0.3588, Loss: 1.371, F1: 0.2311, Score: 0.2732
Epoch:  50
[Epoch 50] Train time: 5.43, Training accuracy: 0.1489, Loss: 1.583, LR: 0.000001
[Epoch 50] Validation accuracy: 0.3461, Loss: 1.374, F1: 0.2050, Score: 0.2516


KeyboardInterrupt: 

In [5]:
num_classes = 4
modeltype = 'large'
model = FSFNet(img_size=112, num_classes=num_classes, type=modeltype)

load_weight 0


In [6]:
model.eval()

FSFNet(
  (backbone): Backbone(
    (input_layer): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): PReLU(num_parameters=64)
    )
    (body): Sequential(
      (0): bottleneck(
        (shortcut_layer): MaxPool2d(kernel_size=1, stride=2, padding=0, dilation=1, ceil_mode=False)
        (res_layer): Sequential(
          (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (2): PReLU(num_parameters=64)
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (1): bottleneck(
        (shortcut_layer): MaxPool2d(kernel_size=1, stride=1, padding=0, dilati

In [ ]:
datapath = '/'
train_dataset = ECdataset(datapath, train=True, transform=data_transforms)
val_dataset = ECdataset(datapath, train=False, transform=data_transforms_val)

In [ ]:
val_num = val_dataset.__len__()
print('Train set size:', train_dataset.__len__())
print('Validation set size:', val_dataset.__len__())

Example of fine tuning by frozing all middle layers

In [ ]:
from torchvision import models
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler

# Load the pre-trained ResNet-18 model
model_ft = models.resnet18(weights='IMAGENET1K_V1')

# Freeze all the layers
for param in model_ft.parameters():
    param.requires_grad = False

# Replace the fully connected layer with a new one (for your specific task)
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, 2)

# Ensure the new fully connected layer's parameters are trainable
for param in model_ft.fc.parameters():
    param.requires_grad = True

# Move the model to the desired device (GPU/CPU)
model_ft = model_ft.to(device)

# Define the loss function
criterion = nn.CrossEntropyLoss()

# Observe that only the fully connected layer's parameters are being optimized
optimizer_ft = optim.SGD(model_ft.fc.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)
